# Week 12, Day 1: Optimisation problems: losses, minima and landscapes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(12)
theta = np.repeat(np.linspace(0, 2 * np.pi, 16, endpoint=False), 10)        # 16 directions, 10 trials each
def von_mises(theta, mu, kappa, b=5.0, A=30.0):
    return b + A * np.exp(kappa * (np.cos(theta - mu) - 1))
counts = rng.poisson(von_mises(theta, 1.0, 2.0))                             # true preferred direction 1 rad, width 2
plt.figure(figsize=(5, 3)); plt.plot(np.degrees(theta), counts, ".", alpha=0.4); plt.xlabel("direction (deg)"); plt.ylabel("spike count"); plt.show()

### Exercise: Two losses on a grid

Write `mse(mu, kappa)` and `poisson_nll(mu, kappa)` for the von Mises model. Evaluate both on a grid of preferred directions (0 to 2π, 91 values) and widths (0.5 to 5, 46 values), and store the best (mu, kappa) pair of each as `best_mse` and `best_nll`. Plot the squared-error landscape.

In [ ]:
def mse(mu, kappa):
    return ...

def poisson_nll(mu, kappa):
    lam = von_mises(theta, mu, kappa)
    return ...

mus = np.linspace(0, 2 * np.pi, 91)
kappas = np.linspace(0.5, 5, 46)
L_mse = np.array([[mse(m, k) for m in mus] for k in kappas])
L_nll = np.array([[poisson_nll(m, k) for m in mus] for k in kappas])
i, j = np.unravel_index(np.argmin(L_mse), L_mse.shape); best_mse = (mus[j], kappas[i])
i, j = np.unravel_index(np.argmin(L_nll), L_nll.shape); best_nll = (mus[j], kappas[i])

plt.figure(figsize=(5, 3.2)); plt.contourf(np.degrees(mus), kappas, np.log(L_mse), 30); plt.colorbar(label="log MSE")
plt.plot(np.degrees(best_mse[0]), best_mse[1], "w*", ms=12); plt.xlabel("preferred direction (deg)"); plt.ylabel("width κ"); plt.show()
print("best by MSE:", np.round(best_mse, 3), "  best by Poisson NLL:", np.round(best_nll, 3), "  truth: (1.0, 2.0)")

In [ ]:
import numpy as np
def test_losses():
    "mse and poisson_nll compute the two losses"
    lam = von_mises(theta, 0.5, 1.5)
    assert np.isclose(mse(0.5, 1.5), np.mean((counts - lam) ** 2))
    assert np.isclose(poisson_nll(0.5, 1.5), np.mean(lam - counts * np.log(lam)))

def test_grid():
    "both grid minima are close to the true parameters"
    for m, k in [best_mse, best_nll]:
        assert abs(m - 1.0) < 0.1 and abs(k - 2.0) < 0.5

test_losses()
test_grid()
print("All checks passed")

### Exercise: Local search on a rugged landscape

Write `freq_loss(f)`. Evaluate it on a fine grid from 1 to 20 Hz, count the local minima (`n_local_minima`) and find the grid minimum (`f_grid`). Then run a simple local search from 18 starts between 2 and 19 Hz (500 steps of f ← f − 0.5 × numerical slope) and store the fraction that end within 0.2 Hz of 7.3 Hz in `success`.

In [ ]:
fs = 200
tt = np.arange(0, 1, 1 / fs)
xx = np.cos(2 * np.pi * 7.3 * tt + 0.5) + 0.5 * rng.normal(size=len(tt))

def freq_loss(f):
    B = np.column_stack([np.cos(2 * np.pi * f * tt), np.sin(2 * np.pi * f * tt)])
    coef = np.linalg.lstsq(B, xx, rcond=None)[0]
    return ...

fgrid = np.linspace(1, 20, 1901)
Lf = np.array([freq_loss(f) for f in fgrid])
n_local_minima = int(np.sum((Lf[1:-1] < Lf[:-2]) & (Lf[1:-1] < Lf[2:])))
f_grid = ...

ends = []
for f in np.linspace(2, 19, 18):
    for _ in range(500):
        slope = (freq_loss(f + 1e-4) - freq_loss(f - 1e-4)) / 2e-4
        f = ...
    ends.append(f)
success = np.mean(np.abs(np.array(ends) - 7.3) < 0.2)

plt.figure(figsize=(6, 3)); plt.plot(fgrid, Lf); plt.plot(ends, [freq_loss(f) for f in ends], "ro", ms=4)
plt.xlabel("candidate frequency (Hz)"); plt.ylabel("loss"); plt.show()
print(n_local_minima, "local minima; grid minimum at", f_grid, "Hz;  local searches that succeeded:", round(success, 2))

In [ ]:
import numpy as np
def test_loss():
    "freq_loss is the mean squared residual after the best cosine-sine fit at f"
    B = np.column_stack([np.cos(2 * np.pi * 5 * tt), np.sin(2 * np.pi * 5 * tt)])
    c = np.linalg.lstsq(B, xx, rcond=None)[0]
    assert np.isclose(freq_loss(5.0), np.mean((xx - B @ c) ** 2))

def test_minima():
    "the landscape has many local minima, and the grid minimum is the true frequency"
    assert n_local_minima > 8 and abs(f_grid - 7.3) < 0.1

def test_local_search():
    "fewer than a third of the 18 local searches find the true frequency"
    assert success < 1 / 3

test_loss()
test_minima()
test_local_search()
print("All checks passed")